In [29]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import threading
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import numpy as np
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy import stats
import random
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
random.seed(42)

In [30]:
df=pd.read_csv('/content/D1_UK_Beans_Brand_PPG.csv')
df.fillna(0,inplace=True)

unique_ppg=df.PPG.unique()
unique_ppg

array(['Others', 'Standard Multi', 'Standard Single', 'Small Single',
       'Small Multi', 'Small SNAP POTS'], dtype=object)

In [31]:
def remove_outliers_iqr(df):

    numeric_cols = df.select_dtypes(include='number').columns
    # numeric_cols=['SalesValue']
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filter out outliers for the current column
        df = df[~((df[col] < lower_bound) | (df[col] > upper_bound))]

    return df

In [32]:
list_df=[]

In [33]:
unique_channel=df.Channel.unique()
for ch in unique_channel:
  df_ch=df[df['Channel']==ch]

  unique_ppg=df_ch.PPG.unique()
  # unique_ppg=['Standard Multi']
  for ppg in unique_ppg:
    df_ch_ppg=df_ch[df_ch['PPG']==ppg]
    unique_brand=df_ch_ppg.Brand.unique()


    df_ch_ppg=df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
          'Variant', 'PackType', 'PPG', 'PackSize', 'Year',
          'Month', 'Week', 'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1',
          'PPU', 'PPL', 'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality',
          'Trend', 'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann',
          'D1_Comp@Direct', 'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct',
          'PPU_Comp@Indirect', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect',
          'Up_Down_Index', 'Cat_Pr_Index']]
    df_ch_ppg.dropna(inplace=True)
    df_ch_ppg.reset_index(drop=True,inplace=True)
    # df_ch_ppg=remove_outliers_iqr(df_ch_ppg)
    if len(df_ch_ppg)>30:


      print(f'enough data points( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')
      list_df.append(df_ch_ppg)

enough data points( 255   ) for combination of Channel>>> Convenience and PPG >>> Others and available brands are ['BRANSTON' 'HEINZ Standard' 'Restofcategory'] 
enough data points( 492   ) for combination of Channel>>> Convenience and PPG >>> Standard Multi and available brands are ['BRANSTON' 'HEINZ Standard' 'Private Label Std'] 
enough data points( 817   ) for combination of Channel>>> Convenience and PPG >>> Standard Single and available brands are ['BRANSTON' 'HEINZ Flavoured' 'HEINZ Standard' 'Private Label Std'
 'Restofcategory'] 
enough data points( 475   ) for combination of Channel>>> Convenience and PPG >>> Small Single and available brands are ['HEINZ Flavoured' 'HEINZ Standard' 'Private Label Std' 'Restofcategory'] 
enough data points( 164   ) for combination of Channel>>> Convenience and PPG >>> Small Multi and available brands are ['HEINZ Standard'] 
enough data points( 164   ) for combination of Channel>>> Convenience and PPG >>> Small SNAP POTS and available brands ar

In [42]:
df_ch_ppg=remove_outliers_iqr(list_df[3])


In [43]:
columns_in_use = ['PPL', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'D1', 'D1_cann',
                      'D1_Comp@Direct', 'D1_Comp@Indirect', 'Seasonality',
                      'Trend', 'Up_Down_Index', 'Cat_Pr_Index']
df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
df_encoded = df_encoded.replace({True: 1, False: 0})
relative_pr=['PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect']
for col in relative_pr:
      df_encoded[col] = np.where(
          df_encoded[col] == 0,  # Condition: if the column value is 0
          df_encoded[col],       # Keep it the same (unchanged)
          df_encoded['PPL'] / df_encoded[col]  )# Else: compute the ratio

# Get unique brand and encoded column names
unique_brand = df_ch_ppg['Brand'].unique()
len_unique_brand = len(unique_brand)
df_encoded_columns = list(df_encoded.columns)
encoded_columns = df_encoded_columns[-len_unique_brand:]
for col in encoded_columns:
        df_encoded[f'PPL_{col}'] = df_encoded['PPL'] * df_encoded[col]

# Columns used for modeling (including interaction terms)
interaction_columns = [f'PPL_{col}' for col in encoded_columns]
columns_taken = [item for sublist in [columns_in_use, encoded_columns, interaction_columns] for item in sublist]
y_var = df_encoded['Volume']

In [44]:
x_full = df_encoded[columns_taken]
x_full
x_train, x_test, y_train, y_test = train_test_split(x_full, y_var, test_size=0.2, random_state=42)

train_indices, test_indices = train_test_split(df_encoded.index, test_size=0.2, random_state=42)


print("Train indices:", train_indices)

print("Test indices:", test_indices)


Train indices: Index([454, 420,  37, 334, 320, 108,  41, 311,  20, 403,
       ...
        98, 399, 293, 474,  24, 367,  81, 278, 466, 231],
      dtype='int64', length=220)
Test indices: Index([ 34, 296, 375, 299, 402, 422, 316, 387, 450, 315,  90, 304, 425, 461,
       324, 314, 456,  69,  95, 427,  49,  84, 359, 341, 460, 395, 289,  46,
        13, 312,  26, 281,  28, 292, 352,  86, 419, 421,  10,  77,  50, 298,
        75,  29, 286, 391,  23, 385, 345, 388,  97, 331, 336,  19, 335,  14],
      dtype='int64')


In [65]:
# Prediction function (including intercept)
def predict(X, W):
    output = X.dot(W[1:]) + W[0] #This line and all other lines inside the function should be indented.
    relu_output = np.maximum(0, output)  # Apply ReLU
    # print(relu_output)
    return output

# Define the cost function (including L2 regularization)
# def cost_function(params, X, Y, l2_penalty):
#     W = params
#     Y_pred = predict(X, W)
#     cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
#     return cost
def cost_function(params, X, Y, l2_penalty):
    """
    Cost function to minimize MAPE with L2 regularization.

    Args:
        params (np.array): Weight vector (including bias).
        X (np.array): Feature matrix.
        Y (np.array): True target values.
        l2_penalty (float): L2 regularization penalty.

    Returns:
        float: Cost (MAPE + L2 penalty).
    """
    W = params
    # Predict target values
    Y_pred = predict(X, W)

    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-8

    # Calculate MAPE
    mape = np.mean(np.abs((Y - Y_pred) / (Y + epsilon))) * 100  # In percentage

    # Add L2 penalty (excluding bias term, i.e., W[1:])
    l2_penalty_term = l2_penalty * np.sum(W[1:] ** 2)

    # Combine MAPE and L2 penalty
    cost = mape + l2_penalty_term

    return cost

# MAPE calculation function
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
    weighted_mape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)
    return weighted_mape * 100
# def get_constraints(feature_names):
#     constraints = [{'type': 'ineq', 'fun': lambda params: -params[1] - 0.01}]
#     for i, name in enumerate(feature_names):
#         if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
#             constraints.append({'type': 'ineq', 'fun': lambda params, i=i: params[i]})
#         # elif name in ['D1_Comp@Direct', 'D1_cann']:
#         #     constraints.append({'type': 'ineq', 'fun': lambda params, i=i: -params[i] - 0.00001})
#     return constraints

def get_constraints():
    constraints = [
        {'type': 'ineq', 'fun': lambda params: -params[1] - 0.01},  # Ensure param[1] < -0.01 (negative constraint)
        {'type': 'ineq', 'fun': lambda params: -params[2]-0.01},          # Ensure param[2] > 0 (positive constraint)
        {'type': 'ineq', 'fun': lambda params: -params[3]-0.01},          # Ensure param[3] > 0 (positive constraint)
        {'type': 'ineq', 'fun': lambda params: -params[4]-0.01},          # Ensure param[4] > 0 (positive constraint)
    ]
    return constraints


In [67]:
 # List of L2 penalty values to iterate over
l2_penalty_list = [0.01, 0.1, 1, 10, 100]

# Initialize variables to store the best results
mape_custom_train_prev = float('inf')
best_l2_penalty = None
best_W_opt = None
best_y_pred_custom_train = None
best_y_pred_custom_test = None

for l2_penalty in l2_penalty_list:
    # Initial parameters (weights + intercept)
    initial_params = np.zeros(x_train.shape[1] + 1)

    # Get the constraints
    constraints = get_constraints()

    # Minimize the cost function using SLSQP
    result = minimize(
        cost_function,
        initial_params,
        constraints=constraints,
        args=(x_train, y_train, l2_penalty),
        method='SLSQP'
    )

    # Extract optimized parameters
    W_opt = result.x

    # Make predictions on the training set
    y_pred_custom_train = predict(x_train, W_opt)
    y_pred_custom_test = predict(x_test, W_opt)

    # Calculate MAPE for the custom implementation
    mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)
    print(f"MAPE for L2 penalty {l2_penalty}: {mape_custom_train_new}")
    if mape_custom_train_new < mape_custom_train_prev:
        mape_custom_train_prev = mape_custom_train_new
        best_l2_penalty = l2_penalty
        best_W_opt = W_opt
        best_y_pred_custom_train = y_pred_custom_train
        best_y_pred_custom_test = y_pred_custom_test

MAPE for L2 penalty 0.01: 74.86234490871273
MAPE for L2 penalty 0.1: 110.4867076352069
MAPE for L2 penalty 1: 97.66844438344548
MAPE for L2 penalty 10: 99.69130147346057
MAPE for L2 penalty 100: 99.74015097894238


In [68]:
best_W_opt

array([ 1.69651027e+00, -7.61962358e+00, -7.57205235e+00, -1.16374161e+00,
       -5.46186586e+00,  1.79161768e+01, -1.88729452e-02, -3.21908309e-01,
        7.19078880e-01, -5.62947583e-07,  1.07882889e-01, -6.50516125e-01,
       -1.12414602e+00,  2.40380416e+00,  1.30517604e-02,  6.71554639e-01,
       -3.74941194e+00,  1.18719902e+01,  5.12248778e-02,  1.18393594e+00,
        2.08444680e+01])

In [69]:
df_ch_ppg

,Market,Channel,Region,Category,SubCategory,Brand,Variant,PackType,PPG,PackSize,...,D1_Comp@Direct,D1_Comp@Indirect,PPU_cann,PPU_Comp@Direct,PPU_Comp@Indirect,PPL_cann,PPL_Comp@Direct,PPL_Comp@Indirect,Up_Down_Index,Cat_Pr_Index
4,UK,Convenience,AllRegion,Beans,AllSubCategory,HEINZ Flavoured,All,All,Small Single,All,...,80.208507,80.067069,1.146880,0.598842,1.066200,2.819698,2.951072,1.493168,2.171502,1.678102
5,UK,Convenience,AllRegion,Beans,AllSubCategory,HEINZ Flavoured,All,All,Small Single,All,...,83.153618,79.514260,1.144491,0.566931,1.020202,2.816248,2.793812,1.501724,2.207861,1.636920
6,UK,Convenience,AllRegion,Beans,AllSubCategory,HEINZ Flavoured,All,All,Small Single,All,...,83.405393,78.119958,1.132811,0.552695,0.984470,2.785448,2.723654,1.500259,2.242242,1.606734
7,UK,Convenience,AllRegion,Beans,AllSubCategory,HEINZ Flavoured,All,All,Small Single,All,...,83.885693,78.276404,1.140519,0.553827,0.985455,2.810036,2.729232,1.500580,2.247091,1.610216
8,UK,Convenience,AllRegion,Beans,AllSubCategory,HEINZ Flavoured,All,All,Small Single,All,...,84.389501,80.170554,1.144879,0.553555,1.056316,2.816599,2.728830,1.511092,2.238805,1.656043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
467,UK,Convenience,AllRegion,Beans,AllSubCategory,Restofcategory,All,All,Small Single,All,...,90.389916,83.492877,0.962582,1.042300,1.279156,2.446299,5.247778,2.030171,2.178194,2.347641
468,UK,Convenience,AllRegion,Beans,AllSubCategory,Restofcategory,All,All,Small Single,All,...,90.584611,83.088641,0.880286,1.065390,1.297576,2.236891,5.339571,2.100696,2.193242,2.360832
472,UK,Convenience,AllRegion,Beans,AllSubCategory,Restofcategory,All,All,Small Single,All,...,86.659665,82.030736,0.898612,0.901241,1.428065,2.259645,4.475949,2.256100,2.147250,2.639770
473,UK,Convenience,AllRegion,Beans,AllSubCategory,Restofcategory,All,All,Small Single,All,...,82.975509,80.130146,0.405716,0.852870,1.417729,1.020355,4.173099,2.334649,2.211376,2.592663


In [70]:
y_pred_custom_test

,0
34,3.815278
296,8.181617
375,5.266634
299,1.293381
402,17.201806
422,15.042441
316,1.393659
387,11.356541
450,-1.545763
315,0.116039


In [71]:
x_train

,PPL,PPL_cann,PPL_Comp@Direct,PPL_Comp@Indirect,D1,D1_cann,D1_Comp@Direct,D1_Comp@Indirect,Seasonality,Trend,Up_Down_Index,Cat_Pr_Index,Brand_HEINZ Flavoured,Brand_HEINZ Standard,Brand_Private Label Std,Brand_Restofcategory,PPL_Brand_HEINZ Flavoured,PPL_Brand_HEINZ Standard,PPL_Brand_Private Label Std,PPL_Brand_Restofcategory
454,1.136364,0.448817,0.321631,0.596320,0.083537,18.106666,86.527531,78.361102,5.771199e+07,17,2.272504,2.077886,0,0,0,1,0.000000,0.000000,0.000000,1.136364
420,2.099326,1.624375,0.457891,0.794192,69.543967,75.932513,90.782813,79.355039,6.421892e+07,147,2.226750,2.427972,0,0,1,0,0.000000,0.000000,2.099326,0.000000
37,7.400035,2.527117,2.247050,4.695154,12.658607,14.972785,88.879660,78.739133,5.661336e+07,38,2.221742,1.744488,1,0,0,0,7.400035,0.000000,0.000000,0.000000
334,1.744132,1.672242,0.471862,0.858557,21.087991,78.751829,91.578521,79.403659,5.659984e+07,61,2.279985,1.926518,0,0,1,0,0.000000,0.000000,1.744132,0.000000
320,1.358849,1.700625,0.370827,0.721538,12.039604,84.278332,93.480433,83.048168,5.570913e+07,47,2.205260,1.830625,0,0,1,0,0.000000,0.000000,1.358849,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,1.738815,1.642182,0.321621,0.686409,15.159670,79.756332,90.291683,74.037165,6.421892e+07,94,2.112514,2.406490,0,0,1,0,0.000000,0.000000,1.738815,0.000000
81,3.910774,0.952370,0.809502,1.751643,7.757831,18.185177,82.761431,76.851528,6.117368e+07,82,2.145451,2.490575,1,0,0,0,3.910774,0.000000,0.000000,0.000000
278,1.151827,1.493693,0.338440,0.663937,26.962492,78.610332,93.235063,80.060407,7.639503e+07,5,2.171502,1.678102,0,0,1,0,0.000000,0.000000,1.151827,0.000000
466,1.500000,0.576275,0.293728,0.735467,0.034205,1.771121,88.904982,82.611198,6.620043e+07,29,2.180787,2.355726,0,0,0,1,0.000000,0.000000,0.000000,1.500000


In [72]:
train_indices

Index([454, 420,  37, 334, 320, 108,  41, 311,  20, 403,
       ...
        98, 399, 293, 474,  24, 367,  81, 278, 466, 231],
      dtype='int64', length=220)

In [73]:
K=x_train.iloc[:,4:].dot(best_W_opt[5:])+best_W_opt[0]
three_col= x_train.iloc[:,1:4]
ppl_col=x_train['PPL']

three_col

,PPL_cann,PPL_Comp@Direct,PPL_Comp@Indirect
454,0.448817,0.321631,0.596320
420,1.624375,0.457891,0.794192
37,2.527117,2.247050,4.695154
334,1.672242,0.471862,0.858557
320,1.700625,0.370827,0.721538
...,...,...,...
367,1.642182,0.321621,0.686409
81,0.952370,0.809502,1.751643
278,1.493693,0.338440,0.663937
466,0.576275,0.293728,0.735467


In [74]:
for col in three_col.columns:
  three_col[col]=three_col[col]/ppl_col
three_col

,PPL_cann,PPL_Comp@Direct,PPL_Comp@Indirect
454,0.394959,0.283035,0.524761
420,0.773760,0.218113,0.378308
37,0.341501,0.303654,0.634477
334,0.958782,0.270543,0.492255
320,1.251518,0.272898,0.530992
...,...,...,...
367,0.944426,0.184966,0.394757
81,0.243525,0.206993,0.447902
278,1.296803,0.293829,0.576420
466,0.384183,0.195818,0.490311


In [75]:
new_slope=three_col.dot(best_W_opt[2:5])+best_W_opt[1]
new_slope
mcv=-K/new_slope
mcv.mean()

29.515285314912244

In [76]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Assuming your columns_in_use, df_encoded, and other variables are already defined

# MAPE calculation function
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
    weighted_mape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)
    return weighted_mape * 100

# Function to separate data and calculate MAPE for each brand
def calculate_mape_by_brand(x_train, x_test, y_train, y_test, df_encoded, columns_taken,best_W_opt):
    # Get the list of brand columns (these are the one-hot encoded columns)
    encoded_columns = [col for col in df_encoded.columns if col.startswith('Brand_')]

    mape_results = {}  # Dictionary to store MAPE for each brand
    mcv_store=[]
    ppl_store=[]
    for col in encoded_columns:
        # Filter data where the column value is 1 (i.e., data for that brand)
        x_train_brand = x_train[x_train[col] == 1]
        x_test_brand = x_test[x_test[col] == 1]
        y_train_brand = y_train[x_train[col] == 1]
        y_test_brand = y_test[x_test[col] == 1]

        # If no data points for that brand in either train or test, skip
        if x_train_brand.empty or x_test_brand.empty:
            continue



        # Make predictions on train and test
        y_pred_train = predict(x_train_brand,best_W_opt)
        y_pred_test = predict(x_test_brand,best_W_opt)

        # Calculate MAPE for the train and test sets
        mape_train = mean_absolute_percentage_error(y_train_brand, y_pred_train)
        mape_test = mean_absolute_percentage_error(y_test_brand, y_pred_test)

        # Store results in the dictionary
        mape_results[col] = {'train_mape': mape_train, 'test_mape': mape_test}


        K=x_train_brand.iloc[:,4:].dot(best_W_opt[5:])+best_W_opt[0]
        three_col= x_train_brand.iloc[:,1:4]
        ppl_col=x_train_brand['PPL']
        mean_ppl=ppl_col.mean()
        for col in three_col.columns:
            three_col[col]=three_col[col]/ppl_col

        new_slope=three_col.dot(best_W_opt[2:5])+best_W_opt[1]
        new_slope
        mcv=-K/new_slope
        mean_mcv=mcv.mean()
        mcv_store.append(mean_mcv)
        ppl_store.append(mean_ppl)

    return mape_results,mcv_store,ppl_store




In [77]:
# Call the function to get MAPE for each brand
mape_results,mcv_store,ppl_store = calculate_mape_by_brand(x_train, x_test, y_train, y_test, df_encoded, columns_taken,best_W_opt)
brand_list,mape_list_train_brand,mape_list_test_brand=[],[],[]
# Print MAPE results
for brand, mape in mape_results.items():
    print(f"Results for {brand}:")
    print(f"Train MAPE: {mape['train_mape']:.2f}%")
    print(f"Test MAPE: {mape['test_mape']:.2f}%")
    print("-" * 40)
    brand_list.append(brand)
    mape_list_train_brand.append(f"{mape['train_mape']:.2f}%")
    mape_list_test_brand.append(f"{mape['test_mape']:.2f}%")

Results for Brand_HEINZ Flavoured:
Train MAPE: 30.25%
Test MAPE: 35.27%
----------------------------------------
Results for Brand_Private Label Std:
Train MAPE: 71.83%
Test MAPE: 72.03%
----------------------------------------
Results for Brand_Restofcategory:
Train MAPE: 152.02%
Test MAPE: 163.25%
----------------------------------------


In [78]:
for i in range(len(ppl_store)):
  csf=mcv_store[i]/ppl_store[i]
  print(f'CSF for {brand_list[i]} is {csf}')

CSF for Brand_HEINZ Flavoured is 2.80232820515223
CSF for Brand_Private Label Std is 21.25688989822106
CSF for Brand_Restofcategory is 1.0278952472582423


In [21]:
brand_list,mape_list_train_brand,mape_list_test_brand

(['Brand_BRANSTON', 'Brand_HEINZ Standard', 'Brand_Private Label Std'],
 ['22.80%', '14.51%', '77.73%'],
 ['19.83%', '15.67%', '78.31%'])

In [22]:
# Assuming the column names are like 'Brand_BRANSTON', 'Brand_HEINZ', etc., and 'Volume' column is called 'PPL'

# Identify the one-hot encoded columns related to brands
one_hot_columns = [col for col in x_train.columns if col.startswith('Brand_')]

# Initialize an empty dictionary to store the data
separated_data_dict = {}

# Assuming you have a 'Volume' column, adjust this to your actual column name if needed
volume_column = 'Volume'  # or replace 'PPL' with the name of your volume column

# Filter rows for each one-hot encoded column and store in the dictionary along with corresponding volume
for col in one_hot_columns:
    # Filter rows where the one-hot encoded column is 1
    filtered_df = x_train[x_train[col] == 1]

    # Extract the corresponding volume column (PPL or your volume column)
    volume_data = y_train[volume_column]

    # Store both the filtered data and volume data in the dictionary
    separated_data_dict[col] = {
        'filtered_data': filtered_df,
        'volume_data': volume_data
    }

# Display the results (example)
for brand, data in separated_data_dict.items():
    print(f"Data for {brand}:")
    print(data['filtered_data'])
    print(f"Volume for {brand}:")
    print(data['volume_data'])
    print("-" * 40)


KeyError: 'Volume'

In [ ]:
# Identify the one-hot encoded columns
one_hot_columns = [col for col in x_train.columns if col.startswith('Brand_')]

# Dictionary to store filtered DataFrames
filtered_data = {}

# Filter rows for each one-hot encoded column
for col in one_hot_columns:
    filtered_data[col] = x_train[x_train[col] == 1]

# Display results

for col, df in filtered_data.items():
    print(f"Data points where {col} == 1:")
    print(df)



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize

# Define functions
def predict(X, W):
    """Prediction function, temporarily without ReLU for debugging."""
    output = X.dot(W[1:]) + W[0]
    return output

def cost_function(params, X, Y, l2_penalty):
    """Cost function with L2 regularization."""
    W = params
    Y_pred = predict(X, W)
    cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
    return cost

def mean_absolute_percentage_error(y_true, y_pred):
    """Calculate MAPE."""
    y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)  # Replace zero values to avoid division errors
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mape


# List of L2 penalty values to iterate over
l2_penalty_list = [0.01, 0.1, 1, 10, 100]

# Initialize variables to store the best results
mape_custom_train_prev = float('inf')
best_l2_penalty = None
best_W_opt = None
best_y_pred_custom_train = None
best_y_pred_custom_test = None

# Debugging outputs
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

# Loop through L2 penalties
for l2_penalty in l2_penalty_list:
    # Initial parameters (weights + intercept)
    initial_params = np.zeros(x_train.shape[1] + 1)

    # Minimize the cost function using SLSQP
    result = minimize(
        cost_function,
        initial_params,
        args=(x_train, y_train, l2_penalty),
        method='SLSQP'
    )

    # Extract optimized parameters
    W_opt = result.x

    # Debugging outputs
    print(f"\nL2 Penalty: {l2_penalty}")
    print("Optimized Weights:", W_opt)
    print("Optimization success:", result.success)
    print("Optimization message:", result.message)

    # Make predictions on train and test sets
    y_pred_custom_train = predict(x_train, W_opt)
    y_pred_custom_test = predict(x_test, W_opt)

    # Debugging outputs
    print("Sample Predictions (Train):", y_pred_custom_train[:5])
    print("Sample Predictions (Test):", y_pred_custom_test[:5])
    print("Actual Values (Train):", y_train[:5])
    print("Actual Values (Test):", y_test[:5])

    # Calculate MAPE
    mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)
    mape_custom_test_new = mean_absolute_percentage_error(y_test, y_pred_custom_test)
    print(f"MAPE (Train): {mape_custom_train_new:.2f}%")
    print(f"MAPE (Test): {mape_custom_test_new:.2f}%")

    # Update best results
    if mape_custom_train_new < mape_custom_train_prev:
        mape_custom_train_prev = mape_custom_train_new
        best_l2_penalty = l2_penalty
        best_W_opt = W_opt
        best_y_pred_custom_train = y_pred_custom_train
        best_y_pred_custom_test = y_pred_custom_test

# Report the final results
print("\nBest Results:")
print(f"Best L2 Penalty: {best_l2_penalty}")
print(f"Best MAPE (Train): {mape_custom_train_prev:.2f}%")
print(f"Best MAPE (Test): {mean_absolute_percentage_error(y_test, best_y_pred_custom_test):.2f}%")


In [ ]:
df_encoded

In [ ]:
def custom_ridge(df):
